# 如何使用护栏（Guardrails）

在本 notebook 中，我们分享一些为 LLM 应用实现护栏的示例。护栏是一个泛称，指旨在引导应用行为的**检测性控制措施**。由于 LLM 固有的随机性，更强的可控性是一项常见需求，因此在将 LLM 从原型推向生产的过程中，构建有效的护栏已成为最常见的性能优化领域之一。

护栏的形式极其[多样](https://github.com/NVIDIA/NeMo-Guardrails/blob/main/examples/README.md)，几乎可以部署在你能想到 LLM 可能出问题的任何场景中。本 notebook 旨在提供一些简单的示例，你可以对其进行扩展以满足自己独特的用例；同时也会概述在决定是否实现护栏以及如何实现时需要考虑的权衡。

本 notebook 将重点介绍：
1. **输入护栏**：在不当内容到达 LLM 之前将其标记出来
2. **输出护栏**：在 LLM 生成的内容到达客户之前对其进行校验

**注意：** 本 notebook 将护栏作为围绕 LLM 的检测性控制措施的泛称来讨论——如需提供预构建护栏框架的官方库，请查看以下项目：
- [NeMo Guardrails](https://github.com/NVIDIA/NeMo-Guardrails/tree/main)
- [Guardrails AI](https://github.com/ShreyaR/guardrails)


In [20]:
import openai

GPT_MODEL = 'gpt-4o-mini'

## 1. 输入护栏

输入护栏旨在从源头上阻止不当内容到达 LLM——一些常见的用例包括：
- **话题护栏：** 识别用户何时提出了偏离主题的问题，并告诉他们 LLM 可以在哪些话题上提供帮助。
- **越狱（Jailbreaking）：** 检测用户何时试图劫持 LLM 并覆盖其提示词。
- **提示词注入（Prompt injection）：** 发现提示词注入的情况，即用户试图隐藏恶意代码，使其在 LLM 执行的任何下游函数中被执行。

在所有这些情况下，护栏都充当预防性控制措施，在 LLM 之前或与其并行运行，并在满足其中某个条件时触发应用采取不同的行为。

### 设计护栏

设计护栏时，重要的是权衡**准确率**、**延迟**和**成本**，力求以对收益和用户体验最小的影响获得最高的准确率。

我们先从一个简单的**话题护栏**开始，它旨在检测偏离主题的问题，并在触发时阻止 LLM 作答。这个护栏由一个简单的提示词构成，使用 `gpt-4o-mini`，在保持足够准确率的同时最大程度地优化延迟/成本；但如果想进一步优化，可以考虑：
- **准确率：** 可以考虑微调 `gpt-4o-mini` 或使用少样本示例来提高准确率。如果你有一个可以帮助判断某段内容是否被允许的信息语料库，RAG 也会很有效。
- **延迟/成本：** 可以尝试微调更小的模型，例如 `babbage-002`，或 Llama 等开源模型，在提供足够训练样本的情况下，它们的表现可能相当不错。使用开源模型时，你还可以调整用于推理的机器，以最大程度地降低成本或延迟。

这个简单的护栏旨在确保 LLM 只回答一组预定义的话题，并对超出范围的查询返回一条预设消息。

### 拥抱异步

一种常见的降低延迟的设计是：将护栏与主 LLM 调用一起异步发送。如果护栏被触发，就返回护栏的响应；否则返回 LLM 的响应。

我们将采用这种方法，创建一个 `execute_chat_with_guardrails` 函数，它会并行运行 LLM 的 `get_chat_response` 和 `topical_guardrail` 护栏，并且只有在护栏返回 `allowed` 时才返回 LLM 的响应。

### 局限性

在设计时，你应始终考虑护栏的局限性。需要注意的几个关键局限包括：
- 使用 LLM 作为护栏时，请注意它们与你的基础 LLM 调用本身具有相同的漏洞。例如，一次**提示词注入**尝试可能同时绕过你的护栏和实际的 LLM 调用。
- 随着对话变长，你的指令会被额外的文本稀释，LLM 也更容易被**越狱**。
- 如果为了弥补上述问题而把护栏设置得过于严格，可能会损害用户体验。这表现为**过度拒绝**：护栏因为与提示词注入或越狱尝试存在相似之处，而拒绝了无害的用户请求。

### 缓解措施

如果能将护栏与基于规则的检测或更传统的机器学习检测模型结合起来，就可以缓解其中一些风险。我们还看到有客户的护栏只考虑最新一条消息，以降低模型被长对话混淆的风险。

我们还建议采用渐进式发布，并主动监控对话，以便发现提示词注入或越狱的情况，然后添加更多护栏来覆盖这些新类型的行为，或者将它们作为训练样本加入现有护栏。

In [21]:
system_prompt = "You are a helpful assistant."

bad_request = "I want to talk about horses"
good_request = "What are the best breeds of dog for people that like cats?"

In [22]:
import asyncio


async def get_chat_response(user_request):
    print("Getting LLM response")
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_request},
    ]
    response = openai.chat.completions.create(
        model=GPT_MODEL, messages=messages, temperature=0.5
    )
    print("Got LLM response")

    return response.choices[0].message.content


async def topical_guardrail(user_request):
    print("Checking topical guardrail")
    messages = [
        {
            "role": "system",
            "content": "Your role is to assess whether the user question is allowed or not. The allowed topics are cats and dogs. If the topic is allowed, say 'allowed' otherwise say 'not_allowed'",
        },
        {"role": "user", "content": user_request},
    ]
    response = openai.chat.completions.create(
        model=GPT_MODEL, messages=messages, temperature=0
    )

    print("Got guardrail response")
    return response.choices[0].message.content


async def execute_chat_with_guardrail(user_request):
    topical_guardrail_task = asyncio.create_task(topical_guardrail(user_request))
    chat_task = asyncio.create_task(get_chat_response(user_request))

    while True:
        done, _ = await asyncio.wait(
            [topical_guardrail_task, chat_task], return_when=asyncio.FIRST_COMPLETED
        )
        if topical_guardrail_task in done:
            guardrail_response = topical_guardrail_task.result()
            if guardrail_response == "not_allowed":
                chat_task.cancel()
                print("Topical guardrail triggered")
                return "I can only talk about cats and dogs, the best animals that ever lived."
            elif chat_task in done:
                chat_response = chat_task.result()
                return chat_response
        else:
            await asyncio.sleep(0.1)  # sleep for a bit before checking the tasks again

In [23]:
# Call the main function with the good request - this should go through
response = await execute_chat_with_guardrail(good_request)
print(response)

Checking topical guardrail
Got guardrail response
Getting LLM response
Got LLM response
If you like cats and are considering getting a dog, there are several breeds known for their compatibility with feline friends. Here are some of the best dog breeds that tend to get along well with cats:

1. **Golden Retriever**: Friendly and tolerant, Golden Retrievers often get along well with other animals, including cats.

2. **Labrador Retriever**: Similar to Golden Retrievers, Labs are social and friendly, making them good companions for cats.

3. **Cavalier King Charles Spaniel**: This breed is gentle and affectionate, often forming strong bonds with other pets.

4. **Basset Hound**: Basset Hounds are laid-back and generally have a calm demeanor, which can help them coexist peacefully with cats.

5. **Beagle**: Beagles are friendly and sociable, and they often enjoy the company of other animals, including cats.

6. **Pug**: Pugs are known for their playful and friendly nature, which can make 

In [24]:
# Call the main function with the bad request - this should get blocked
response = await execute_chat_with_guardrail(bad_request)
print(response)

Checking topical guardrail
Got guardrail response
Getting LLM response
Got LLM response
Topical guardrail triggered
I can only talk about cats and dogs, the best animals that ever lived.


看起来我们的护栏生效了——第一个问题被放行，而第二个问题因偏离主题被拦截。接下来，我们将把这个概念扩展到对 LLM 返回的响应进行审核。

## 2. 输出护栏

输出护栏管控 LLM 返回的内容。它们可以有多种形式，其中最常见的包括：
- **幻觉/事实核查护栏：** 使用标准事实信息语料库或包含幻觉响应的训练集来拦截幻觉响应。
- **审核护栏：** 应用品牌和企业准则来审核 LLM 的结果，如果违反准则，则拦截或改写其响应。
- **语法检查：** LLM 返回的结构化输出可能已损坏或无法解析——这类护栏会检测到这些情况，并进行重试或优雅地失败，从而防止下游应用出错。
    - 这是函数调用中常用的一种控制措施，确保 LLM 返回 `function_call` 时，`arguments` 中返回的是预期的 schema。
    
### 审核护栏

这里我们实现一个**审核护栏**，它使用 [G-Eval](https://arxiv.org/abs/2303.16634) 评估方法的一个版本，对 LLM 响应中不良内容的存在程度进行评分。我们在另一个 [notebook](https://github.com/openai/openai-cookbook/blob/main/examples/evaluation/How_to_eval_abstractive_summarization.ipynb) 中更详细地演示了这种方法。

为此，我们将构建一个可扩展的内容审核框架：它接收一个 `domain`，并使用一组 `steps` 将 `criteria` 应用于一段 `content`：
1. 我们设置一个领域名称，描述要审核的内容类型。
2. 我们提供标准，清楚地说明内容应该包含和不应该包含什么。
3. 为 LLM 提供逐步的评分指令。
4. LLM 返回 1-5 之间的离散分数。

### 设置护栏阈值

我们的输出护栏会评估 LLM 的响应，并拦截所有得分为 3 或更高的内容。设置这个阈值是常见的优化环节——我们建议构建一个评估集，并使用混淆矩阵对结果进行评分，从而为护栏设定合适的容忍度。这里的权衡通常是：
- 更多的误报会导致割裂的用户体验，让客户感到烦恼，助手也显得不那么有用。
- 更多的漏报可能对你的业务造成持久的损害，因为有人会让助手回答不当问题，或对其进行提示词注入/越狱。

例如，对于越狱，你可能希望设置非常低的阈值，因为如果你的 LLM 被劫持并被用来生成危险内容，最终流传到社交媒体上，对你的业务风险非常高。不过，对于我们的用例，我们愿意接受少量漏报，因为最坏的情况不过是有人最终养了一只比熊犬，而他其实更适合养拉布拉多——这虽然令人遗憾，但（我们希望）大概不会对我们的业务造成持久的损害。

In [16]:
domain = "animal breed recommendation"

animal_advice_criteria = """
Assess the presence of explicit recommendation of cat or dog breeds in the content.
The content should contain only general advice about cats and dogs, not specific breeds to purchase."""

animal_advice_steps = """
1. Read the content and the criteria carefully.
2. Assess how much explicit recommendation of cat or dog breeds is contained in the content.
3. Assign an animal advice score from 1 to 5, with 1 being no explicit cat or dog breed advice, and 5 being multiple named cat or dog breeds.
"""

moderation_system_prompt = """
You are a moderation assistant. Your role is to detect content about {domain} in the text provided, and mark the severity of that content.

## {domain}

### Criteria

{scoring_criteria}

### Instructions

{scoring_steps}

### Content

{content}

### Evaluation (score only!)
"""

In [17]:
async def moderation_guardrail(chat_response):
    print("Checking moderation guardrail")
    mod_messages = [
        {"role": "user", "content": moderation_system_prompt.format(
            domain=domain,
            scoring_criteria=animal_advice_criteria,
            scoring_steps=animal_advice_steps,
            content=chat_response
        )},
    ]
    response = openai.chat.completions.create(
        model=GPT_MODEL, messages=mod_messages, temperature=0
    )
    print("Got moderation response")
    return response.choices[0].message.content
    
    
async def execute_all_guardrails(user_request):
    topical_guardrail_task = asyncio.create_task(topical_guardrail(user_request))
    chat_task = asyncio.create_task(get_chat_response(user_request))

    while True:
        done, _ = await asyncio.wait(
            [topical_guardrail_task, chat_task], return_when=asyncio.FIRST_COMPLETED
        )
        if topical_guardrail_task in done:
            guardrail_response = topical_guardrail_task.result()
            if guardrail_response == "not_allowed":
                chat_task.cancel()
                print("Topical guardrail triggered")
                return "I can only talk about cats and dogs, the best animals that ever lived."
            elif chat_task in done:
                chat_response = chat_task.result()
                moderation_response = await moderation_guardrail(chat_response)

                if int(moderation_response) >= 3:
                    print(f"Moderation guardrail flagged with a score of {int(moderation_response)}")
                    return "Sorry, we're not permitted to give animal breed advice. I can help you with any general queries you might have."

                else:
                    print('Passed moderation')
                    return chat_response
        else:
            await asyncio.sleep(0.1)  # sleep for a bit before checking the tasks again

In [18]:
# Adding a request that should pass both our topical guardrail and our moderation guardrail
great_request = 'What is some advice you can give to a new dog owner?'

In [19]:
tests = [good_request,bad_request,great_request]

for test in tests:
    result = await execute_all_guardrails(test)
    print(result)
    print('\n\n')
    

Checking topical guardrail
Got guardrail response
Getting LLM response
Got LLM response
Checking moderation guardrail
Got moderation response
Moderation guardrail flagged with a score of 5
Sorry, we're not permitted to give animal breed advice. I can help you with any general queries you might have.



Checking topical guardrail
Got guardrail response
Getting LLM response
Got LLM response
Topical guardrail triggered
I can only talk about cats and dogs, the best animals that ever lived.



Checking topical guardrail
Got guardrail response
Getting LLM response
Got LLM response
Checking moderation guardrail
Got moderation response
Moderation guardrail flagged with a score of 3
Sorry, we're not permitted to give animal breed advice. I can help you with any general queries you might have.





## 结论

护栏是 LLM 领域中一个活跃且不断发展的话题，希望本 notebook 能为你有效介绍护栏的核心概念。回顾一下：
- 护栏是一种检测性控制措施，旨在防止有害内容到达你的应用和用户，并为生产环境中的 LLM 增加可控性。
- 护栏可以采用输入护栏的形式（在内容到达 LLM 之前进行处理），也可以采用输出护栏的形式（控制 LLM 的响应）。
- 设计护栏并设置其阈值是在准确率、延迟和成本之间进行权衡。你的决策应基于对护栏性能的清晰评估，以及对漏报和误报给业务带来的代价的理解。
- 通过采用异步设计原则，你可以横向扩展护栏，从而在护栏数量和覆盖范围增加时，将对用户的影响降到最低。

我们期待看到你如何进一步运用这些内容，以及随着生态系统的成熟，关于护栏的思考将如何演进。